In [0]:
# Define Parameters
dbutils.widgets.text("catalog", "dbr_dev", "1. Unity Catalog Name")
dbutils.widgets.text("secret_scope", "valerii-matviiv-scope", "2. Secret Scope Name")
dbutils.widgets.text("secret_key", "finnhub-api-key", "3. Finnhub API Key Name")

# Retrieve Parameter Values
catalog = dbutils.widgets.get("catalog")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")

username = "valeriimatviiv"
bronze_schema = f"{username}_bronze"
silver_schema = f"{username}_silver"
gold_schema = f"{username}_gold"
volume_name = "market_radar_landing"

# Verify Secret Scope Connection
try:
    api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)
    print(f"Successfully connected to secret scope: '{secret_scope}'")
except Exception as e:
    raise RuntimeError(f"Failed to fetch secret '{secret_key}' from scope '{secret_scope}': {e}")

# Create Schemas under Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

# Create Shared Landing Volume in Bronze
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{bronze_schema}.{volume_name}")

print(f"Schemas created: {bronze_schema}, {silver_schema}, {gold_schema}")
print(f"Landing Volume created: {catalog}.{bronze_schema}.{volume_name}")

# Establish Base Path Structure for Data Ingestion & Auto Loader State
base_volume_path = f"/Volumes/{catalog}/{bronze_schema}/{volume_name}"

# Raw Landing Paths
landing_price_path = f"{base_volume_path}/landing/nasdaq_price"
landing_news_path = f"{base_volume_path}/landing/finnhub_news"

# Auto Loader Checkpoint Paths
checkpoint_price_path = f"{base_volume_path}/_state/checkpoints/nasdaq_price"
checkpoint_news_path = f"{base_volume_path}/_state/checkpoints/finnhub_news"

# Auto Loader Schema Evolution Paths
schema_price_path = f"{base_volume_path}/_state/schemas/nasdaq_price"
schema_news_path = f"{base_volume_path}/_state/schemas/finnhub_news"

# Create directories on Volume
dbutils.fs.mkdirs(landing_price_path)
dbutils.fs.mkdirs(landing_news_path)
dbutils.fs.mkdirs(checkpoint_price_path)
dbutils.fs.mkdirs(checkpoint_news_path)
dbutils.fs.mkdirs(schema_price_path)
dbutils.fs.mkdirs(schema_news_path)

print("All landing, checkpoint, and schema directories successfully initialized.")

In [0]:
# # Verify folder hierarchy on Volume
# base_volume_path = f"/Volumes/{dbutils.widgets.get('catalog')}/valeriimatviiv_bronze/market_radar_landing"

# print("--- Volume Directory Contents ---")
# display(dbutils.fs.ls(base_volume_path))

# print("--- Landing Paths ---")
# display(dbutils.fs.ls(f"{base_volume_path}/landing"))